# Exp2-v2 — BoW + 전처리 + 핸드크래프트 + W&B Sweep

- 목적: Exp2(BoW)에서 Exp3 개선 아이디어(전처리, handcraft feature)를 이식해 성능 향상 확인
- 제약: MLP 구조 유지

In [ ]:
!pip install datasets wandb scikit-learn -q

In [ ]:
import re, copy, numpy as np
import torch, torch.nn as nn, torch.optim as optim, torch.backends.cudnn as cudnn
from datasets import load_dataset
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import accuracy_score
from scipy.sparse import hstack, csr_matrix
import wandb

SEED=42
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
cudnn.benchmark=False; cudnn.deterministic=True
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device:{device}')

In [ ]:
data=load_dataset('Sp1786/multiclass-sentiment-analysis-dataset')
def remove_empty(row):
    return all(row[f] not in [None,''] for f in ['id','text','label','sentiment'])
train_data=data['train'].filter(remove_empty)
dev_data=data['validation'].filter(remove_empty)
test_data=data['test'].filter(remove_empty)
output_size=len(set(train_data['label']))
train_labels=train_data['label']
test_labels_list=test_data['label']
print(f"Train:{len(train_data)} | Dev:{len(dev_data)} | Test:{len(test_data)} | Classes:{output_size}")

In [ ]:
def preprocess_text(text):
    text = str(text).lower()
    text = text.replace('`', "'")
    text = text.replace('****', ' bad ').replace('***', ' bad ')
    text = re.sub(r"won't", "will not", text)
    text = re.sub(r"can't", "cannot", text)
    text = re.sub(r"n't", " not", text)
    text = re.sub(r"'re", " are", text)
    text = re.sub(r"'ve", " have", text)
    text = re.sub(r"'ll", " will", text)
    text = re.sub(r"'d", " would", text)
    text = re.sub(r"'m", " am", text)
    for pat, rep in [
        (r'\\bidk\\b','i do not know'), (r'\\bur\\b','your'),
        (r'\\bnaw\\b','no'), (r'\\bgonna\\b','going to'),
        (r'\\bwanna\\b','want to'), (r'\\blol\\b','laughing'),
        (r'\\bomg\\b','oh my god'), (r'\\bwtf\\b','what the'),
    ]:
        text = re.sub(pat, rep, text)
    return text

def extract_handcraft(texts):
    feats=[]
    for text in texts:
        t=str(text)
        tl=t.lower()
        words=t.split()
        feats.append([
            min(t.count('!'),5),
            min(t.count('?'),5),
            sum(1 for w in words if w.isupper() and len(w)>1),
            min(len(words),50),
            int('http' in tl),
            int(any(e in tl for e in [':)',':(',':d',':/','haha','hehe','lmao']))
        ])
    return np.array(feats, dtype=np.float32)

vectorizer=CountVectorizer(max_features=30000, preprocessor=preprocess_text, min_df=2)
vectorizer.fit(train_data['text'])

def build_features(split):
    bow=vectorizer.transform(split['text'])
    hc=csr_matrix(extract_handcraft(split['text']))
    x=hstack([bow,hc])
    return torch.FloatTensor(x.toarray()).to(device)

train_t=build_features(train_data)
dev_t=build_features(dev_data)
test_t=build_features(test_data)
dev_labels_t=torch.tensor(dev_data['label'],dtype=torch.long).to(device)
input_size=train_t.shape[1]
print(f'Input size: {input_size}')

In [ ]:
class MLP(nn.Module):
    def __init__(self,i,h,o,d=0.0):
        super().__init__()
        self.fc1=nn.Linear(i,h)
        self.fc2=nn.Linear(h,h//2)
        self.fc3=nn.Linear(h//2,o)
        self.activation=nn.GELU()
        self.output_act=nn.Softmax(dim=1)
        self.dropout=nn.Dropout(p=d)
    def forward(self,x):
        x=self.dropout(self.activation(self.fc1(x)))
        x=self.dropout(self.activation(self.fc2(x)))
        return self.output_act(self.fc3(x))

In [ ]:
sweep_config={
    'method':'bayes',
    'metric':{'name':'best_dev_accuracy','goal':'maximize'},
    'parameters':{
        'hidden_size':{'values':[256,512,1000]},
        'learning_rate':{'distribution':'log_uniform_values','min':1e-5,'max':1e-3},
        'dropout':{'values':[0.1,0.2,0.3,0.4]},
        'weight_decay':{'values':[0,1e-5,1e-4]},
        'batch_size':{'values':[128,256]},
        'num_epochs':{'values':[30,50]},
    }
}
sweep_id=wandb.sweep(sweep_config, project='nlp-hw1')
print(f'Sweep ID: {sweep_id}')

In [ ]:
def train_sweep():
    run=wandb.init()
    cfg=run.config
    torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
    model=MLP(input_size,cfg.hidden_size,output_size,cfg.dropout).to(device)
    opt=optim.Adam(model.parameters(),lr=cfg.learning_rate,weight_decay=cfg.weight_decay)
    lfn=nn.CrossEntropyLoss()
    best_dev,best_state=0,None

    for epoch in range(cfg.num_epochs):
        model.train()
        total_loss=0
        for i in range(0,len(train_t),cfg.batch_size):
            bd=train_t[i:i+cfg.batch_size]
            bl=torch.tensor(train_labels[i:i+cfg.batch_size],device=device)
            loss=lfn(model(bd),bl)
            opt.zero_grad(); loss.backward(); opt.step()
            total_loss+=loss.item()

        model.eval()
        with torch.no_grad():
            da=(torch.argmax(model(dev_t),dim=1)==dev_labels_t).float().mean().item()
        if da>best_dev:
            best_dev,best_state=da,copy.deepcopy(model.state_dict())

        wandb.log({
            'epoch':epoch+1,
            'train_loss':total_loss/max(1,len(train_t)//cfg.batch_size),
            'dev_accuracy':da,
            'best_dev_accuracy':best_dev,
        })

    model.load_state_dict(best_state)
    with torch.no_grad():
        test_acc=accuracy_score(test_labels_list,torch.argmax(model(test_t),dim=1).cpu().tolist())
    wandb.log({'test_accuracy':test_acc})
    print(f"[Exp2-v2] Dev:{best_dev:.4f}|Test:{test_acc*100:.2f}%")
    wandb.finish()

wandb.agent(sweep_id, train_sweep, count=12)